# 1. MongoDB Aggregation Framework

The **Aggregation Framework** is MongoDB's powerful data processing pipeline. It allows you to:
- Transform documents
- Group and summarize data
- Perform complex calculations
- Join collections (like SQL JOINs)

## Pipeline Concept

```
Documents -> Stage 1 -> Stage 2 -> Stage 3 -> Results
                $match    $group    $sort
```

Each **stage** transforms the documents and passes them to the next stage.

# 2. Setup

In [ ]:
from pymongo import MongoClient
from datetime import datetime, timedelta
import pprint

# Connect
client = MongoClient("mongodb://localhost:27017/")
db = client["aggregation_demo"]

# Collections
orders = db["orders"]
products = db["products"]
users = db["users"]

# Clear existing data
orders.delete_many({})
products.delete_many({})
users.delete_many({})

def show(result, title=""):
    if title:
        print(f"\n{title}")
        print("=" * 60)
    for doc in result:
        pprint.pprint(doc)

print("Setup complete!")

In [ ]:
# Insert sample data

# Users
users.insert_many([
    {"_id": 1, "name": "John", "city": "New York", "age": 30},
    {"_id": 2, "name": "Jane", "city": "Boston", "age": 25},
    {"_id": 3, "name": "Bob", "city": "New York", "age": 35},
    {"_id": 4, "name": "Alice", "city": "Chicago", "age": 28},
])

# Products
products.insert_many([
    {"_id": 101, "name": "Laptop", "category": "Electronics", "price": 999},
    {"_id": 102, "name": "Phone", "category": "Electronics", "price": 699},
    {"_id": 103, "name": "Desk", "category": "Furniture", "price": 299},
    {"_id": 104, "name": "Chair", "category": "Furniture", "price": 149},
    {"_id": 105, "name": "Headphones", "category": "Electronics", "price": 199},
])

# Orders
orders.insert_many([
    {"user_id": 1, "product_id": 101, "quantity": 1, "total": 999, "status": "delivered", "date": datetime(2024, 1, 15)},
    {"user_id": 1, "product_id": 105, "quantity": 2, "total": 398, "status": "delivered", "date": datetime(2024, 1, 20)},
    {"user_id": 2, "product_id": 102, "quantity": 1, "total": 699, "status": "shipped", "date": datetime(2024, 2, 1)},
    {"user_id": 2, "product_id": 103, "quantity": 1, "total": 299, "status": "delivered", "date": datetime(2024, 2, 5)},
    {"user_id": 3, "product_id": 101, "quantity": 2, "total": 1998, "status": "pending", "date": datetime(2024, 2, 10)},
    {"user_id": 3, "product_id": 104, "quantity": 4, "total": 596, "status": "delivered", "date": datetime(2024, 2, 15)},
    {"user_id": 4, "product_id": 102, "quantity": 1, "total": 699, "status": "delivered", "date": datetime(2024, 2, 20)},
    {"user_id": 1, "product_id": 103, "quantity": 1, "total": 299, "status": "shipped", "date": datetime(2024, 2, 25)},
])

print(f"Inserted: {users.count_documents({})} users, {products.count_documents({})} products, {orders.count_documents({})} orders")

# 3. Basic Aggregation Stages

## 3.1 $match - Filtering

In [ ]:
# $match - Filter documents (like find())

pipeline = [
    {"$match": {"status": "delivered"}}
]

result = orders.aggregate(pipeline)
show(result, "Delivered orders")

In [ ]:
# $match with multiple conditions

pipeline = [
    {"$match": {
        "status": "delivered",
        "total": {"$gte": 500}
    }}
]

result = orders.aggregate(pipeline)
show(result, "Delivered orders >= $500")

## 3.2 $project - Reshaping

In [ ]:
# $project - Select/rename fields, create computed fields

pipeline = [
    {"$project": {
        "_id": 0,              # Exclude _id
        "user_id": 1,          # Include user_id
        "total": 1,            # Include total
        "order_status": "$status",  # Rename status to order_status
        "tax": {"$multiply": ["$total", 0.08]},  # Computed field
    }}
]

result = orders.aggregate(pipeline)
show(result, "Projected orders with tax")

## 3.3 $group - Grouping and Aggregating

In [ ]:
# $group - Group by a field and aggregate

pipeline = [
    {"$group": {
        "_id": "$user_id",           # Group by user_id
        "total_orders": {"$sum": 1},  # Count orders
        "total_spent": {"$sum": "$total"},  # Sum of totals
        "avg_order": {"$avg": "$total"},    # Average order
    }}
]

result = orders.aggregate(pipeline)
show(result, "Orders grouped by user")

In [ ]:
# $group by status with multiple aggregations

pipeline = [
    {"$group": {
        "_id": "$status",
        "count": {"$sum": 1},
        "total_value": {"$sum": "$total"},
        "min_order": {"$min": "$total"},
        "max_order": {"$max": "$total"},
        "orders": {"$push": "$total"}  # Collect values in array
    }}
]

result = orders.aggregate(pipeline)
show(result, "Orders by status")

In [ ]:
# $group for overall totals (use null for _id)

pipeline = [
    {"$group": {
        "_id": None,  # No grouping - aggregate all
        "total_orders": {"$sum": 1},
        "total_revenue": {"$sum": "$total"},
        "avg_order_value": {"$avg": "$total"}
    }}
]

result = orders.aggregate(pipeline)
show(result, "Overall order statistics")

## 3.4 $sort and $limit

In [ ]:
# $sort - Sort results
# $limit - Limit results

pipeline = [
    {"$group": {
        "_id": "$user_id",
        "total_spent": {"$sum": "$total"}
    }},
    {"$sort": {"total_spent": -1}},  # -1 = descending
    {"$limit": 3}  # Top 3
]

result = orders.aggregate(pipeline)
show(result, "Top 3 customers by spending")

## 3.5 $unwind - Flatten Arrays

In [ ]:
# First, add a document with an array
orders.insert_one({
    "user_id": 5,
    "items": [
        {"product": "Laptop", "qty": 1, "price": 999},
        {"product": "Mouse", "qty": 2, "price": 25},
        {"product": "Keyboard", "qty": 1, "price": 75}
    ],
    "total": 1124,
    "status": "pending"
})

# $unwind - Create separate document for each array element
pipeline = [
    {"$match": {"user_id": 5}},
    {"$unwind": "$items"},
    {"$project": {
        "_id": 0,
        "product": "$items.product",
        "qty": "$items.qty",
        "price": "$items.price",
        "line_total": {"$multiply": ["$items.qty", "$items.price"]}
    }}
]

result = orders.aggregate(pipeline)
show(result, "Unwound order items")

# 4. $lookup - Joining Collections

In [ ]:
# $lookup - Join with another collection (like SQL JOIN)

pipeline = [
    {"$match": {"user_id": {"$in": [1, 2]}}},  # Filter first
    {"$lookup": {
        "from": "users",        # Collection to join
        "localField": "user_id",    # Field in orders
        "foreignField": "_id",      # Field in users
        "as": "user_info"           # Output array field name
    }},
    {"$unwind": "$user_info"},  # Flatten the array
    {"$project": {
        "_id": 0,
        "order_total": "$total",
        "status": 1,
        "customer_name": "$user_info.name",
        "customer_city": "$user_info.city"
    }}
]

result = orders.aggregate(pipeline)
show(result, "Orders with user info (JOIN)")

In [ ]:
# Multiple $lookups - Join multiple collections

pipeline = [
    {"$match": {"status": "delivered", "product_id": {"$exists": True}}},
    # Join with users
    {"$lookup": {
        "from": "users",
        "localField": "user_id",
        "foreignField": "_id",
        "as": "user"
    }},
    # Join with products
    {"$lookup": {
        "from": "products",
        "localField": "product_id",
        "foreignField": "_id",
        "as": "product"
    }},
    {"$unwind": "$user"},
    {"$unwind": "$product"},
    {"$project": {
        "_id": 0,
        "customer": "$user.name",
        "product": "$product.name",
        "category": "$product.category",
        "quantity": 1,
        "total": 1
    }},
    {"$limit": 5}
]

result = orders.aggregate(pipeline)
show(result, "Orders with user AND product info")

# 5. Aggregation Operators

## 5.1 Arithmetic Operators

In [ ]:
# Arithmetic operators in $project

pipeline = [
    {"$match": {"product_id": {"$exists": True}}},
    {"$project": {
        "_id": 0,
        "total": 1,
        "quantity": 1,
        # $add - Addition
        "total_with_shipping": {"$add": ["$total", 10]},
        # $subtract - Subtraction
        "discount": {"$subtract": ["$total", {"$multiply": ["$total", 0.1]}]},
        # $multiply - Multiplication
        "tax": {"$multiply": ["$total", 0.08]},
        # $divide - Division
        "unit_price": {"$divide": ["$total", "$quantity"]},
        # $mod - Modulus
        "remainder": {"$mod": ["$total", 100]}
    }},
    {"$limit": 3}
]

result = orders.aggregate(pipeline)
show(result, "Arithmetic operators")

## 5.2 String Operators

In [ ]:
# String operators

pipeline = [
    {"$project": {
        "_id": 0,
        "name": 1,
        # $toUpper - Uppercase
        "name_upper": {"$toUpper": "$name"},
        # $toLower - Lowercase
        "name_lower": {"$toLower": "$name"},
        # $concat - Concatenate strings
        "greeting": {"$concat": ["Hello, ", "$name", "!"]},
        # $substr - Substring (deprecated, use $substrCP)
        "initials": {"$substrCP": ["$name", 0, 1]},
        # $strLenCP - String length
        "name_length": {"$strLenCP": "$name"}
    }}
]

result = users.aggregate(pipeline)
show(result, "String operators")

## 5.3 Date Operators

In [ ]:
# Date operators

pipeline = [
    {"$match": {"date": {"$exists": True}}},
    {"$project": {
        "_id": 0,
        "date": 1,
        "total": 1,
        # Extract date parts
        "year": {"$year": "$date"},
        "month": {"$month": "$date"},
        "day": {"$dayOfMonth": "$date"},
        "dayOfWeek": {"$dayOfWeek": "$date"},  # 1=Sunday, 7=Saturday
        "week": {"$week": "$date"},
        # Format date as string
        "formatted": {
            "$dateToString": {
                "format": "%Y-%m-%d",
                "date": "$date"
            }
        }
    }},
    {"$limit": 3}
]

result = orders.aggregate(pipeline)
show(result, "Date operators")

In [ ]:
# Group by month

pipeline = [
    {"$match": {"date": {"$exists": True}}},
    {"$group": {
        "_id": {
            "year": {"$year": "$date"},
            "month": {"$month": "$date"}
        },
        "total_sales": {"$sum": "$total"},
        "order_count": {"$sum": 1}
    }},
    {"$sort": {"_id.year": 1, "_id.month": 1}}
]

result = orders.aggregate(pipeline)
show(result, "Monthly sales")

## 5.4 Conditional Operators

In [ ]:
# $cond - If-then-else

pipeline = [
    {"$match": {"product_id": {"$exists": True}}},
    {"$project": {
        "_id": 0,
        "total": 1,
        "status": 1,
        "order_size": {
            "$cond": {
                "if": {"$gte": ["$total", 1000]},
                "then": "Large",
                "else": {
                    "$cond": {
                        "if": {"$gte": ["$total", 500]},
                        "then": "Medium",
                        "else": "Small"
                    }
                }
            }
        }
    }}
]

result = orders.aggregate(pipeline)
show(result, "Orders with size category")

In [ ]:
# $switch - Multiple conditions (like switch/case)

pipeline = [
    {"$project": {
        "_id": 0,
        "name": 1,
        "age": 1,
        "age_group": {
            "$switch": {
                "branches": [
                    {"case": {"$lt": ["$age", 25]}, "then": "Young"},
                    {"case": {"$lt": ["$age", 35]}, "then": "Adult"},
                    {"case": {"$gte": ["$age", 35]}, "then": "Senior"}
                ],
                "default": "Unknown"
            }
        }
    }}
]

result = users.aggregate(pipeline)
show(result, "Users with age groups")

# 6. Advanced Stages

## 6.1 $addFields - Add New Fields

In [ ]:
# $addFields - Add fields without listing all existing fields

pipeline = [
    {"$match": {"product_id": {"$exists": True}}},
    {"$addFields": {
        "tax": {"$multiply": ["$total", 0.08]},
        "final_total": {"$multiply": ["$total", 1.08]}
    }},
    {"$project": {"items": 0, "date": 0}},  # Remove some fields
    {"$limit": 3}
]

result = orders.aggregate(pipeline)
show(result, "Orders with added tax fields")

## 6.2 $bucket - Grouping into Ranges

In [ ]:
# $bucket - Group into custom ranges

pipeline = [
    {"$match": {"total": {"$exists": True}}},
    {"$bucket": {
        "groupBy": "$total",
        "boundaries": [0, 300, 600, 1000, 2000],  # Ranges: 0-299, 300-599, 600-999, 1000-1999
        "default": "Other",  # For values outside boundaries
        "output": {
            "count": {"$sum": 1},
            "total_value": {"$sum": "$total"},
            "orders": {"$push": "$total"}
        }
    }}
]

result = orders.aggregate(pipeline)
show(result, "Orders bucketed by total")

## 6.3 $facet - Multiple Pipelines

In [ ]:
# $facet - Run multiple pipelines in parallel

pipeline = [
    {"$match": {"product_id": {"$exists": True}}},
    {"$facet": {
        "by_status": [
            {"$group": {"_id": "$status", "count": {"$sum": 1}}}
        ],
        "by_user": [
            {"$group": {"_id": "$user_id", "total_spent": {"$sum": "$total"}}},
            {"$sort": {"total_spent": -1}},
            {"$limit": 3}
        ],
        "summary": [
            {"$group": {
                "_id": None,
                "total_orders": {"$sum": 1},
                "total_revenue": {"$sum": "$total"}
            }}
        ]
    }}
]

result = list(orders.aggregate(pipeline))
print("\nFaceted results:")
print("="*60)
print("\nBy Status:")
pprint.pprint(result[0]["by_status"])
print("\nTop 3 Users:")
pprint.pprint(result[0]["by_user"])
print("\nSummary:")
pprint.pprint(result[0]["summary"])

# 7. Indexes

Indexes improve query performance significantly.

In [ ]:
# Create indexes

# Single field index
orders.create_index("user_id")
print("Created index on user_id")

# Compound index (multiple fields)
orders.create_index([("status", 1), ("date", -1)])
print("Created compound index on status + date")

# Unique index
users.create_index("name", unique=True)
print("Created unique index on name")

# List all indexes
print("\nIndexes on orders collection:")
for index in orders.list_indexes():
    print(f"  {index['name']}: {index['key']}")

In [ ]:
# Text index for full-text search

products.create_index([("name", "text"), ("category", "text")])

# Use text search
result = products.find({"$text": {"$search": "laptop electronics"}})
show(result, "Text search for 'laptop electronics'")

In [ ]:
# Drop an index
# orders.drop_index("user_id_1")

# Drop all indexes (except _id)
# orders.drop_indexes()

print("Index management examples (commented for safety)")

# 8. Real-World Aggregation Examples

In [ ]:
# Example 1: Sales Dashboard Data

pipeline = [
    {"$match": {"date": {"$exists": True}}},
    {"$facet": {
        "total_stats": [
            {"$group": {
                "_id": None,
                "total_revenue": {"$sum": "$total"},
                "total_orders": {"$sum": 1},
                "avg_order_value": {"$avg": "$total"}
            }}
        ],
        "by_status": [
            {"$group": {
                "_id": "$status",
                "count": {"$sum": 1},
                "revenue": {"$sum": "$total"}
            }}
        ],
        "monthly_trend": [
            {"$group": {
                "_id": {"$month": "$date"},
                "revenue": {"$sum": "$total"},
                "orders": {"$sum": 1}
            }},
            {"$sort": {"_id": 1}}
        ]
    }}
]

result = list(orders.aggregate(pipeline))[0]
print("Sales Dashboard Data:")
print("="*60)
pprint.pprint(result)

In [ ]:
# Example 2: Customer Segmentation (RFM Analysis)

pipeline = [
    {"$match": {"date": {"$exists": True}}},
    # Group by user
    {"$group": {
        "_id": "$user_id",
        "total_spent": {"$sum": "$total"},
        "order_count": {"$sum": 1},
        "last_order": {"$max": "$date"},
        "first_order": {"$min": "$date"}
    }},
    # Add customer segment
    {"$addFields": {
        "avg_order_value": {"$divide": ["$total_spent", "$order_count"]},
        "customer_segment": {
            "$switch": {
                "branches": [
                    {"case": {"$gte": ["$total_spent", 1500]}, "then": "Premium"},
                    {"case": {"$gte": ["$total_spent", 500]}, "then": "Regular"},
                ],
                "default": "New"
            }
        }
    }},
    # Join with user details
    {"$lookup": {
        "from": "users",
        "localField": "_id",
        "foreignField": "_id",
        "as": "user"
    }},
    {"$unwind": "$user"},
    {"$project": {
        "_id": 0,
        "customer_name": "$user.name",
        "city": "$user.city",
        "total_spent": {"$round": ["$total_spent", 2]},
        "order_count": 1,
        "avg_order_value": {"$round": ["$avg_order_value", 2]},
        "customer_segment": 1
    }},
    {"$sort": {"total_spent": -1}}
]

result = orders.aggregate(pipeline)
show(result, "Customer Segmentation")

# 9. Summary

## Key Aggregation Stages

| Stage | Description |
|-------|-------------|
| `$match` | Filter documents |
| `$project` | Reshape documents, select fields |
| `$group` | Group and aggregate |
| `$sort` | Sort documents |
| `$limit` / `$skip` | Pagination |
| `$unwind` | Flatten arrays |
| `$lookup` | Join collections |
| `$addFields` | Add computed fields |
| `$bucket` | Group into ranges |
| `$facet` | Multiple pipelines |

## Key Operators

| Category | Operators |
|----------|----------|
| **Arithmetic** | `$add`, `$subtract`, `$multiply`, `$divide` |
| **Comparison** | `$eq`, `$gt`, `$lt`, `$cond`, `$switch` |
| **String** | `$concat`, `$toUpper`, `$toLower`, `$substr` |
| **Date** | `$year`, `$month`, `$dayOfMonth`, `$dateToString` |
| **Array** | `$push`, `$addToSet`, `$first`, `$last` |
| **Accumulator** | `$sum`, `$avg`, `$min`, `$max`, `$count` |

## Best Practices

1. **Use `$match` early** - Filter before processing
2. **Use indexes** - For `$match` and `$sort` stages
3. **Limit fields** - Use `$project` to reduce data
4. **Order matters** - Pipeline stages execute in order

In [ ]:
# Cleanup
client.close()
print("Connection closed.")